In [43]:
import pandas as pd
import numpy as np
import json

# INE datasets inspection

This notebook provides a sequential inspection of datasets selected from INE Portugal, focusing on their structure, data quality, temporal and geographic granularity, value ranges, units, and consistency across datasets.

## **income_inhabitant_0012670**: Gross reported  income per inhabitant (€) by Geographic localization (NUTS - 2024)

In [44]:
with open(r"../data_raw/Income_inhabitant_0012670.json", "r", encoding='utf-8') as file:
    data = json.load(file)

data = data[0]['Dados']['2024'] # Al tener el diccionario en una lista, se accede a ella mediante [0]
data

[{'geocod': '11C1804',
  'geodsg': 'Cinfães',
  'ind_string': '7 696',
  'valor': '7696'},
 {'geocod': '11C1813',
  'geodsg': 'Resende',
  'ind_string': '7 839',
  'valor': '7839'},
 {'geocod': '1C10211',
  'geodsg': 'Odemira',
  'ind_string': '7 958',
  'valor': '7958'},
 {'geocod': '11B1712',
  'geodsg': 'Valpaços',
  'ind_string': '8 100',
  'valor': '8100'},
 {'geocod': '11D1819',
  'geodsg': 'Tabuaço',
  'ind_string': '8 191',
  'valor': '8191'},
 {'geocod': '11B1709',
  'geodsg': 'Ribeira de Pena',
  'ind_string': '8 214',
  'valor': '8214'},
 {'geocod': '1500803',
  'geodsg': 'Aljezur',
  'ind_string': '8 377',
  'valor': '8377'},
 {'geocod': '11C1302',
  'geodsg': 'Baião',
  'ind_string': '8 403',
  'valor': '8403'},
 {'geocod': '11D1815',
  'geodsg': 'São João da Pesqueira',
  'ind_string': '8 509',
  'valor': '8509'},
 {'geocod': '11B1702',
  'geodsg': 'Boticas',
  'ind_string': '8 525',
  'valor': '8525'},
 {'geocod': '1941803',
  'geodsg': 'Castro Daire',
  'ind_string': '8

In [45]:
income_inhabitant_df = pd.DataFrame(data)
income_inhabitant_df.head(5)

,geocod,geodsg,ind_string,valor
0,11C1804,Cinfães,7 696,7696
1,11C1813,Resende,7 839,7839
2,1C10211,Odemira,7 958,7958
3,11B1712,Valpaços,8 100,8100
4,11D1819,Tabuaço,8 191,8191


In [46]:
income_inhabitant_df = income_inhabitant_df.rename(columns={"valor": "income"})
income_inhabitant_df = income_inhabitant_df.drop(columns=['ind_string'])

In [47]:
income_inhabitant_df.shape

(347, 3)

In [48]:
income_inhabitant_df.dtypes

geocod    object
geodsg    object
income    object
dtype: object

In [49]:
income_inhabitant_df.describe()

,geocod,geodsg,income
count,347,347,347
unique,347,340,331
top,1A01110,Região Autónoma da Madeira,10658
freq,1,3,3


In [50]:
income_inhabitant_df.isna().sum()

geocod    0
geodsg    0
income    0
dtype: int64

In [51]:
income_inhabitant_df.duplicated().sum()

np.int64(0)

In [56]:
(income_inhabitant_df.dropna(subset=['geodsg'])
 .drop_duplicates(subset=['geodsg'])
 .assign(normalized = income_inhabitant_df['geodsg'].astype('string').str.strip().str.lower())
 .groupby('normalized')['geodsg']
 .nunique()
 ).max()

np.int64(1)

In [60]:
income_inhabitant_df.groupby('geocod')['geodsg'].nunique().max()

np.int64(1)

In [61]:
income_inhabitant_df.groupby('geodsg')['geocod'].nunique().max()

np.int64(3)

In [62]:
check = (
    income_inhabitant_df
    .groupby('geodsg')['geocod']
    .nunique()
)

check[check > 1]

conflict_names = check[check > 1].index

income_inhabitant_df[
    income_inhabitant_df['geodsg'].isin(conflict_names)
][['geodsg', 'geocod']].drop_duplicates()

,geodsg,geocod
165,Algarve,15
166,Algarve,150
262,Região Autónoma da Madeira,3
263,Região Autónoma da Madeira,30
264,Região Autónoma da Madeira,300
286,Região Autónoma dos Açores,2
287,Região Autónoma dos Açores,20
288,Região Autónoma dos Açores,200
308,Península de Setúbal,1B
309,Península de Setúbal,1B0


In [52]:
income_inhabitant_df.to_csv(r"../data_raw/income_inhabitant.csv", encoding='utf-8', index=False)

## Observaciones sobre el dataset 'income_inhabitant':


Se ha convertido el dataset de JSON a CSV para facilitar su carga posterior.
Tras los cambios necesarios a nivel de formato del dataset, se ha llevado a cabo la revisión del mismo.

Las unidades indicadas en los metadatos son correctas.
El dataset se compone de 347 registros y 4 variables reducidas a 3 para evitar redundancia.

No se observan valores numéricos extremos.

En cuanto al tipo de datos de las variables, se debería convertir a string tanto 'geocod' como 'geodsg'. 'income' debería ser numérica.

No se observan duplicados. Los nombres de cada zona geográfica mantienen un mismo formato.

Para cada 'geocod' se observa un solo 'geodsg', lo cual es correcto.
Para cada 'geodsg' puede haber más de un 'geocod' si bien se comprueba que se trata de distintos niveles de clasificación geográfica para una misma región, por lo que es correcto.

En cuanto a los códigos: 'geocod' es un código jerárquico NUTS cuya longitud indica el nivel geográfico. En los concelhos (7 caracteres) está formado por NUTS III (3) + código de concelho DICO (4), el mismo código que usa E-REDES. En la limpieza se extrae como 'cod_concelho'. Estas comprobaciones de nombres validan la fuente; los nombres definitivos de cada nivel se asignarán desde la tabla de referencia geográfica ('app/utils/concelho_codes/') a partir de 'cod_concelho'.

## **land_use_variation_0014800**: Rate of surface variation (% - COS - Series 2) of territorial units by Geographic localization (NUTS - 2024) and Surface by land use and cover

CHECK SOURCE

## **pop_density_0013189**: Population's density (No./ km²) by Place of residence (NUTS - 2024)

In [66]:
with open(r"../data_raw/pop_density_0013189.json", "r", encoding='utf-8') as file:
    data = json.load(file)

data = data[0]['Dados']['2025']

pop_density_df = pd.DataFrame(data)
pop_density_df.head(5)

,geocod,geodsg,ind_string,valor
0,1500802,Alcoutim,"4,3",4.3
1,1C20209,Mértola,"4,6",4.6
2,1950505,Idanha-a-Nova,"5,9",5.9
3,1C31203,Avis,"6,0",6
4,1C31211,Monforte,"6,8",6.8


In [67]:
pop_density_df = pop_density_df.drop(columns=['ind_string'])
pop_density_df = pop_density_df.rename(columns={'valor': 'density'})
pop_density_df.head(2)

,geocod,geodsg,density
0,1500802,Alcoutim,4.3
1,1C20209,Mértola,4.6


In [68]:
pop_density_df.shape

(347, 3)

In [69]:
pop_density_df.dtypes

geocod     object
geodsg     object
density    object
dtype: object

In [70]:
pop_density_df.describe()

,geocod,geodsg,density
count,347,347,347
unique,347,340,316
top,1A01115,Região Autónoma dos Açores,7.7
freq,1,3,3


In [71]:
pop_density_df.isna().sum()

geocod     0
geodsg     0
density    0
dtype: int64

In [72]:
pop_density_df.duplicated().sum()

np.int64(0)

In [73]:
(pop_density_df.dropna(subset=['geodsg'])
 .drop_duplicates(subset=['geodsg'])
 .assign(normalized = pop_density_df['geodsg'].astype('string').str.strip().str.lower())
 .groupby('normalized')['geodsg']
 .nunique()
).max()

np.int64(1)

In [74]:
pop_density_df.groupby('geocod')['geodsg'].nunique().max()

np.int64(1)

In [75]:
pop_density_df.groupby('geodsg')['geocod'].nunique().max()

np.int64(3)

In [ ]:
check = pop_density_df.groupby('geodsg')['geocod'].nunique()

conflict_names = check[check > 1].index # Al haber agrupado por geodsg, pasa a ser el index

pop_density_df[pop_density_df['geodsg'].isin(conflict_names)]

,geocod,geodsg,density
204,2,Região Autónoma dos Açores,105.7
205,20,Região Autónoma dos Açores,105.7
206,200,Região Autónoma dos Açores,105.7
213,15,Algarve,115.7
214,150,Algarve,115.7
287,3,Região Autónoma da Madeira,332.2
288,30,Região Autónoma da Madeira,332.2
289,300,Região Autónoma da Madeira,332.2
310,1B,Península de Setúbal,576.9
311,1B0,Península de Setúbal,576.9


In [83]:
pop_density_df.to_csv(r"../data_raw/pop_density.csv", encoding='utf-8', index=False)

## Observaciones sobre el dataset 'pop_density':

Se ha convertido el dataset de JSON a CSV para facilitar su carga posterior.
Tras los cambios necesarios a nivel de formato del dataset, se ha llevado a cabo la revisión del mismo.

Las unidades indicadas en los metadatos son correctas.
El dataset se compone de 347 registros y 4 variables reducidas a 3 para evitar redundancia.

No se observan valores numéricos extremos.

En cuanto al tipo de datos de las variables, se debería convertir a string tanto 'geocod' como 'geodsg'. 'density' debería ser numérica.

No se observan duplicados. Los nombres de cada zona geográfica mantienen un mismo formato.

Para cada 'geocod' se observa un solo 'geodsg', lo cual es correcto.
Para cada 'geodsg' puede haber más de un 'geocod' si bien se comprueba que se trata de distintos niveles de clasificación geográfica para una misma región, por lo que es correcto. Misma situación que el caso anterior.

En ambos casos, las islas quedarán fuera del análisis, puesto que solo se cubrirá la parte continental.

En cuanto a los códigos: 'geocod' es un código jerárquico NUTS cuya longitud indica el nivel geográfico. En los concelhos (7 caracteres) está formado por NUTS III (3) + código de concelho DICO (4), el mismo código que usa E-REDES. En la limpieza se extrae como 'cod_concelho'. Estas comprobaciones de nombres validan la fuente; los nombres definitivos de cada nivel se asignarán desde la tabla de referencia geográfica ('app/utils/concelho_codes/') a partir de 'cod_concelho'.